# Notebook 7 - Effect Size Preservation

Reviewer question: does ComBat harmonization preserve how far pathological subjects deviate from the normative reference population (CamCAN)?

Within a site, ComBat applies the same location/scale correction to every subject, so the patient-vs-HC difference *within* a site is invariant by construction. What can be lost is the deviation of patients *relative to the normative reference*. Every subject is z-scored against a CamCAN normative model (per metric, per bundle); for each filtering method we regress the mean z of patients computed on their harmonized values (`mean_z_harm`) against the mean z computed on their reference, ground-truth values (`mean_z_ref`). The headline numbers are the *unconstrained* OLS fit of that regression - slope, intercept, R² - and the mean absolute error; a perfect harmonization would give slope = 1, intercept = 0, R² = 1, MAE = 0.

`HC` (ComBat fit on the healthy controls only) is the best correction achievable in this setup, not a synonym for "perfect": the supplementary section at the end of this notebook shows that the synthetic site bias also rescales each feature's age effect, which ComBat's location/scale model cannot undo, so even `HC` keeps a residual, age-dependent error. Methods should therefore be read against the `HC` ceiling at a given ratio, not against an absolute target of 1.

Only the `n = 100` synthetic sites are used. Results are cached under `RESULTS/EFFECT_SIZE/classic2/classic` and plots are saved under `RESULTS/EFFECT_SIZE/classic2/classic/PLOTS`.


In [ ]:
import os
import time
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


In [ ]:
SITE_ROOT = "DATA/processed/synthetic_sites/classic/harmonized_AUG_5/ALL"
HARM_ROOT = "RESULTS/MAE_TEST/PROCESS/classic2/classic/ALL"
CAMCAN_DIR = "DATA/processed/CamCAN_clean"

OUT_BASE = "RESULTS/EFFECT_SIZE/classic2/classic"
SITES_OUT_ROOT = f"{OUT_BASE}/ALL"
PLOTS_FOLDER = f"{OUT_BASE}/PLOTS"

METRICS = ["ad", "adt", "afd", "fa", "fat", "fw", "md", "mdt", "rd", "rdt"]

# Only the n = 100 configurations are needed for this analysis.
SAMPLE_SIZES = [100]
DISEASE_RATIOS = [3, 10, 30, 50, 70, 80]

# Bundles present in CamCAN but not in the synthetic site files.
EXCLUDED_CAMCAN_BUNDLES = ["left_ventricle", "right_ventricle"]

# METHOD_DIR (folder name under RESULTS/.../{metric}/) -> display name.
# NB: the `hc` folder stores its harmonized file under a `NoRobust` name too
# (same as the `NoRobust` folder itself), so methods must never be told
# apart by filename, only by the folder they were read from.
METHOD_DIR_TO_NAME = {
    "hc": "HC",
    "NoRobust": "NO_FILTERING",
    "MLP5_ALL": "MLP",
    "MLP7_ALL": "MLP7",
    "G_MAD": "G_MAD",
    "PCA_MD": "PCA_MD",
    "G_ZS": "G_ZS",
    "MAD": "MAD",
    "IQR": "IQR",
    "ZS": "ZS",
    "SN": "SN",
    "QN": "QN",
    "VS": "VS",
    "MMS": "MMS",
}

# Methods shown in the main figures; the cache stores every method found.
MAIN_METHODS = ["HC", "MLP", "NO_FILTERING", "G_MAD", "PCA_MD", "G_ZS", "MAD", "IQR"]

BASE_COLOR_PALETTES = {
    "HC": "chartreuse",
    "NO_FILTERING": "red",
    "MLP": "yellow",
    "PCA_MD": "#37474F",  # dark blue-grey, distinct from the plain greys
}
TICK_FONTSIZE = 20
COLOR_PALETTE = "gray"

FORCE_RECOMPUTE = False
N_JOBS = 8

# Age-diagnostic subset (supplementary section): a handful of sites only, to
# keep the per-subject x bundle error table cheap to cache and re-read.
AGE_DIAGNOSTIC_SITE_IDS = list(range(10))  # site replicate indices 0-9
AGE_DIAGNOSTIC_RATIOS = [3, 80]
AGE_DIAGNOSTIC_METHODS = ["HC", "MLP", "NO_FILTERING"]
AGE_BIN_EDGES = [0, 40, 60, 75, 150]
AGE_BIN_LABELS = ["<40", "40-60", "60-75", "75+"]
AGE_ERROR_PLOT_METRICS = ["fa", "md", "fw", "afd"]
AGE_ERROR_PATH = os.path.join(OUT_BASE, "age_error_subset.csv")

os.makedirs(SITES_OUT_ROOT, exist_ok=True)
os.makedirs(PLOTS_FOLDER, exist_ok=True)


## Normative model (CamCAN)


In [ ]:
def load_camcan(metric):
    """Load the cleaned CamCAN table for one metric.

    Restricted to the 43 bundles also present in the synthetic site files
    (CamCAN additionally has left_ventricle and right_ventricle).
    """
    path = os.path.join(CAMCAN_DIR, f"CamCAN.{metric}.clean.csv.gz")
    camcan = pd.read_csv(path)
    return camcan[~camcan["bundle"].isin(EXCLUDED_CAMCAN_BUNDLES)].copy()


def fit_normative_models(metric):
    """Fit a per-bundle OLS model on CamCAN: mean ~ intercept + sex + handedness + age.

    sex and handedness (coded 1/2 in the data) are used as 0/1 indicators.
    Returns a DataFrame indexed by bundle with the fitted coefficients and
    the residual standard deviation of each model.
    """
    camcan = load_camcan(metric)
    rows = []
    for bundle, data in camcan.groupby("bundle"):
        sex_ind = (data["sex"].to_numpy() == 2).astype(float)
        hand_ind = (data["handedness"].to_numpy() == 2).astype(float)
        age = data["age"].to_numpy()
        design = np.column_stack([sex_ind, hand_ind, age])
        target = data["mean"].to_numpy()

        reg = LinearRegression().fit(design, target)
        residuals = target - reg.predict(design)
        n_params = design.shape[1] + 1  # + intercept
        resid_sd = np.sqrt(np.sum(residuals ** 2) / (len(target) - n_params))

        rows.append(
            {
                "bundle": bundle,
                "intercept": reg.intercept_,
                "coef_sex": reg.coef_[0],
                "coef_handedness": reg.coef_[1],
                "coef_age": reg.coef_[2],
                "resid_sd": resid_sd,
            }
        )
    return pd.DataFrame(rows).set_index("bundle")


NORMATIVE_MODELS = {metric: fit_normative_models(metric) for metric in METRICS}

for metric in METRICS:
    assert len(NORMATIVE_MODELS[metric]) == 43, f"unexpected bundle count for {metric}"
print(f"Fitted normative models for {len(METRICS)} metrics x 43 bundles.")


In [ ]:
def zscore_vs_camcan(df, metric):
    """Add a column 'z' = (mean - predicted) / residual_sd from the CamCAN normative model.

    df: long-format dataframe with columns sid, bundle, mean, age, sex, handedness.
    """
    coefs = NORMATIVE_MODELS[metric]
    merged = df.merge(coefs, left_on="bundle", right_index=True, how="left")

    sex_ind = (merged["sex"].to_numpy() == 2).astype(float)
    hand_ind = (merged["handedness"].to_numpy() == 2).astype(float)
    predicted = (
        merged["intercept"].to_numpy()
        + merged["coef_sex"].to_numpy() * sex_ind
        + merged["coef_handedness"].to_numpy() * hand_ind
        + merged["coef_age"].to_numpy() * merged["age"].to_numpy()
    )

    out = df.copy()
    out["z"] = (merged["mean"].to_numpy() - predicted) / merged["resid_sd"].to_numpy()
    return out


## Site discovery and harmonized file lookup


In [ ]:
def discover_site_ids(sample_size, ratio):
    """List the site replicate indices available for a given (n, ratio) pair."""
    site_dir = os.path.join(SITE_ROOT, f"{sample_size}_{ratio}")
    if not os.path.isdir(site_dir):
        return []
    return sorted(int(name) for name in os.listdir(site_dir) if name.isdigit())


SITE_GRID = [
    (n, ratio, site_id)
    for n in SAMPLE_SIZES
    for ratio in DISEASE_RATIOS
    for site_id in discover_site_ids(n, ratio)
]
print(f"Discovered {len(SITE_GRID)} sites over {len(SAMPLE_SIZES)} sample size(s) x {len(DISEASE_RATIOS)} ratios.")


In [ ]:
def find_harmonized_file(method_dir):
    """Return the harmonized train CSV in method_dir, or None if missing/ambiguous.

    Selection is by exclusion: skip the .model. file, the _test. file and any
    outliers_ file. This is required because the `hc` method folder stores its
    harmonized file under a `NoRobust` name (identical naming to the
    `NoRobust` folder itself), so methods cannot be distinguished by filename.
    """
    if not os.path.isdir(method_dir):
        return None
    candidates = [
        fname
        for fname in os.listdir(method_dir)
        if fname.endswith(".csv")
        and ".model." not in fname
        and "_test." not in fname
        and not fname.startswith("outliers_")
    ]
    if len(candidates) != 1:
        return None
    return os.path.join(method_dir, candidates[0])


## Per-site processing


In [ ]:
EFFECT_COLUMNS = [
    "n", "ratio", "site_id", "metric", "bundle", "method", "group",
    "n_subj", "mean_z_ref", "mean_z_harm", "sd_z_ref", "sd_z_harm",
]
SUMS_COLUMNS = [
    "ratio", "metric", "bundle", "method", "disease",
    "n", "sum_z_ref", "sumsq_z_ref", "sum_z_harm", "sumsq_z_harm",
]


def process_site(n, ratio, site_id):
    """Compute effect-size preservation for one synthetic site (all metrics/methods).

    Caches two files under RESULTS/EFFECT_SIZE/classic2/classic/ALL/{n}_{ratio}/{site_id}/:
    - effect_size_train.csv: per (metric, bundle, method, group in {HC, patients})
      z-score summary, the main analysis table.
    - pathology_sums_train.csv: per (metric, bundle, method, disease) raw sums
      (n, sum z_ref, sum z_ref^2, sum z_harm, sum z_harm^2) used to rebuild the
      pooled RESULTS/EFFECT_SIZE/classic2/classic/pathology_sums.csv without
      re-reading the raw per-subject data on every run.

    Both files are skipped when already present, unless FORCE_RECOMPUTE.
    A method is skipped silently whenever its folder or harmonized file is
    missing for this site.
    """
    site_dir = os.path.join(SITES_OUT_ROOT, f"{n}_{ratio}", str(site_id))
    effect_path = os.path.join(site_dir, "effect_size_train.csv")
    sums_path = os.path.join(site_dir, "pathology_sums_train.csv")

    if not FORCE_RECOMPUTE and os.path.isfile(effect_path) and os.path.isfile(sums_path):
        return pd.read_csv(effect_path), pd.read_csv(sums_path)

    effect_rows = []
    sums_rows = []

    for metric in METRICS:
        gt_path = os.path.join(
            SITE_ROOT,
            f"{n}_{ratio}",
            str(site_id),
            f"gt_train_{n}_{ratio}_{site_id}_{metric}.csv",
        )
        if not os.path.isfile(gt_path):
            continue

        gt = pd.read_csv(gt_path)
        gt = zscore_vs_camcan(gt, metric)
        gt_z = gt[["sid", "bundle", "disease", "z"]].rename(columns={"z": "z_ref"})

        metric_dir = os.path.join(HARM_ROOT, f"{n}_{ratio}", str(site_id), metric)

        for method_dir, method_name in METHOD_DIR_TO_NAME.items():
            harm_file = find_harmonized_file(os.path.join(metric_dir, method_dir))
            if harm_file is None:
                continue

            harm = pd.read_csv(harm_file)
            harm = zscore_vs_camcan(harm, metric)
            harm_z = harm[["sid", "bundle", "z"]].rename(columns={"z": "z_harm"})

            merged = gt_z.merge(harm_z, on=["sid", "bundle"], how="inner")
            if merged.empty:
                continue
            merged["group"] = np.where(merged["disease"] == "HC", "HC", "patients")
            merged["z_ref_sq"] = merged["z_ref"] ** 2
            merged["z_harm_sq"] = merged["z_harm"] ** 2

            summary = (
                merged.groupby(["bundle", "group"])
                .agg(
                    n_subj=("z_ref", "size"),
                    mean_z_ref=("z_ref", "mean"),
                    mean_z_harm=("z_harm", "mean"),
                    sd_z_ref=("z_ref", "std"),
                    sd_z_harm=("z_harm", "std"),
                )
                .reset_index()
            )
            summary["n"] = n
            summary["ratio"] = ratio
            summary["site_id"] = site_id
            summary["metric"] = metric
            summary["method"] = method_name
            effect_rows.append(summary)

            by_disease = (
                merged.groupby(["bundle", "disease"])
                .agg(
                    n=("z_ref", "size"),
                    sum_z_ref=("z_ref", "sum"),
                    sumsq_z_ref=("z_ref_sq", "sum"),
                    sum_z_harm=("z_harm", "sum"),
                    sumsq_z_harm=("z_harm_sq", "sum"),
                )
                .reset_index()
            )
            by_disease["ratio"] = ratio
            by_disease["metric"] = metric
            by_disease["method"] = method_name
            sums_rows.append(by_disease)

    effect_df = (
        pd.concat(effect_rows, ignore_index=True)[EFFECT_COLUMNS]
        if effect_rows
        else pd.DataFrame(columns=EFFECT_COLUMNS)
    )
    sums_df = (
        pd.concat(sums_rows, ignore_index=True)[SUMS_COLUMNS]
        if sums_rows
        else pd.DataFrame(columns=SUMS_COLUMNS)
    )

    os.makedirs(site_dir, exist_ok=True)
    effect_df.to_csv(effect_path, index=False)
    sums_df.to_csv(sums_path, index=False)
    return effect_df, sums_df


## Run the per-site computation


In [ ]:
t0 = time.time()
site_results = Parallel(n_jobs=N_JOBS)(
    delayed(process_site)(n, ratio, site_id) for n, ratio, site_id in SITE_GRID
)
elapsed = time.time() - t0
print(f"Processed {len(site_results)} sites in {elapsed:.1f} s ({elapsed / max(len(site_results), 1):.2f} s/site).")


## Compilation


In [ ]:
def load_all_effect_size():
    """Concatenate every cached effect_size_train.csv under SITES_OUT_ROOT."""
    paths = sorted(glob(os.path.join(SITES_OUT_ROOT, "*", "*", "effect_size_train.csv")))
    frames = [pd.read_csv(path) for path in paths]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=EFFECT_COLUMNS)


def load_all_pathology_sums():
    """Concatenate every cached pathology_sums_train.csv under SITES_OUT_ROOT."""
    paths = sorted(glob(os.path.join(SITES_OUT_ROOT, "*", "*", "pathology_sums_train.csv")))
    frames = [pd.read_csv(path) for path in paths]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=SUMS_COLUMNS)


effect_size_all = load_all_effect_size()
sites_seen = effect_size_all[["ratio", "site_id"]].drop_duplicates().shape[0]
print(f"Loaded {len(effect_size_all):,} rows from {sites_seen} sites.")

nan_rows = int(effect_size_all.isna().any(axis=1).sum())
print(f"Rows with a NaN value: {nan_rows}")

methods_found = sorted(effect_size_all["method"].unique())
print(f"Methods found in the cache: {methods_found}")
missing_main = [method for method in MAIN_METHODS if method not in methods_found]
if missing_main:
    print(f"WARNING: main methods missing from the cache entirely: {missing_main}")


In [ ]:
# Rebuilt from scratch every run: cheap, since it only sums the small
# per-site partial sums cached by process_site (no raw data is re-read).
pathology_sums_parts = load_all_pathology_sums()
pathology_sums = (
    pathology_sums_parts.groupby(["ratio", "metric", "bundle", "method", "disease"], as_index=False)
    .sum(numeric_only=True)
)
pathology_sums.to_csv(os.path.join(OUT_BASE, "pathology_sums.csv"), index=False)
print(f"pathology_sums.csv: {len(pathology_sums):,} rows, diseases = {sorted(pathology_sums['disease'].unique())}")


## Preservation summary tables

Headline metrics: the *unconstrained* OLS slope, intercept, R² and MAE of `mean_z_harm` regressed on `mean_z_ref`, computed over all (site, metric, bundle) rows of the patients group. The through-origin slope (`Σ z_harm·z_ref / Σ z_ref²`) is also stored in the CSV as `through_origin_slope` for reference, but it is confounded by the intercept - a large intercept can pull it close to 1 even when the unconstrained slope is far from 1 - so it is not used in any figure or printed table below.


In [ ]:
def summarize_ols(x, y):
    """Unconstrained OLS fit of y on x: slope, intercept, R^2 and MAE."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    slope = reg.coef_[0]
    intercept = reg.intercept_
    r2 = r2_score(y, reg.predict(x.reshape(-1, 1)))
    mae = np.mean(np.abs(y - x))
    return slope, intercept, r2, mae


def summarize_preservation(group):
    """Headline preservation metrics for one (method, ratio[, metric]) group of patient rows.

    x = mean_z_ref, y = mean_z_harm, one point per (site, metric, bundle) row.
    """
    x = group["mean_z_ref"].to_numpy()
    y = group["mean_z_harm"].to_numpy()
    slope, intercept, r2, mae = summarize_ols(x, y)

    # Kept for reference only (see the markdown cell above): confounded by
    # the intercept, must not be plotted or used as a headline number.
    denom = np.sum(x ** 2)
    through_origin_slope = np.sum(y * x) / denom if denom else np.nan

    return pd.Series(
        {
            "ols_slope": slope,
            "ols_intercept": intercept,
            "r2": r2,
            "mae": mae,
            "through_origin_slope": through_origin_slope,
            "n_rows": len(x),
        }
    )


patients = effect_size_all[effect_size_all["group"] == "patients"]

preservation_by_method_ratio = (
    patients.groupby(["method", "ratio"])
    .apply(summarize_preservation, include_groups=False)
    .reset_index()
)
preservation_by_method_ratio.to_csv(
    os.path.join(OUT_BASE, "preservation_by_method_ratio.csv"), index=False
)

preservation_by_method_ratio_metric = (
    patients.groupby(["method", "ratio", "metric"])
    .apply(summarize_preservation, include_groups=False)
    .reset_index()
)
preservation_by_method_ratio_metric.to_csv(
    os.path.join(OUT_BASE, "preservation_by_method_ratio_metric.csv"), index=False
)

print("preservation_by_method_ratio:", preservation_by_method_ratio.shape)
print("preservation_by_method_ratio_metric:", preservation_by_method_ratio_metric.shape)


In [ ]:
# Variance preservation: median of sd_z_harm / sd_z_ref, for patients and HC.
variance_tmp = effect_size_all.copy()
variance_tmp["sd_ratio"] = variance_tmp["sd_z_harm"] / variance_tmp["sd_z_ref"]
variance_tmp.loc[~np.isfinite(variance_tmp["sd_ratio"]), "sd_ratio"] = np.nan

variance_ratio_by_method_ratio = (
    variance_tmp.groupby(["method", "ratio", "group"])["sd_ratio"]
    .median()
    .reset_index()
    .rename(columns={"sd_ratio": "median_sd_ratio"})
)
variance_ratio_by_method_ratio.to_csv(
    os.path.join(OUT_BASE, "variance_ratio_by_method_ratio.csv"), index=False
)
print("variance_ratio_by_method_ratio:", variance_ratio_by_method_ratio.shape)


## Pathology-level tables

Same OLS decomposition (slope, intercept, R², MAE - not through the origin), computed per (ratio, method, disease) by pooling the 430 metric x bundle features (10 metrics x 43 bundles) of `effect_size_by_pathology.csv`.


In [ ]:
effect_size_by_pathology = pathology_sums.copy()
effect_size_by_pathology["mean_z_ref"] = effect_size_by_pathology["sum_z_ref"] / effect_size_by_pathology["n"]
effect_size_by_pathology["mean_z_harm"] = effect_size_by_pathology["sum_z_harm"] / effect_size_by_pathology["n"]

var_ref = effect_size_by_pathology["sumsq_z_ref"] / effect_size_by_pathology["n"] - effect_size_by_pathology["mean_z_ref"] ** 2
var_harm = effect_size_by_pathology["sumsq_z_harm"] / effect_size_by_pathology["n"] - effect_size_by_pathology["mean_z_harm"] ** 2
effect_size_by_pathology["sd_z_ref"] = np.sqrt(np.clip(var_ref, 0, None))
effect_size_by_pathology["sd_z_harm"] = np.sqrt(np.clip(var_harm, 0, None))

effect_size_by_pathology = effect_size_by_pathology[
    ["ratio", "metric", "bundle", "method", "disease", "n", "mean_z_ref", "mean_z_harm", "sd_z_ref", "sd_z_harm"]
]
effect_size_by_pathology.to_csv(os.path.join(OUT_BASE, "effect_size_by_pathology.csv"), index=False)
print("effect_size_by_pathology:", effect_size_by_pathology.shape)


def summarize_pathology_preservation(group):
    """OLS slope, intercept, R^2 and MAE of pooled mean_z_harm vs pooled mean_z_ref,
    across the (metric, bundle) features pooled for one (ratio, method, disease).
    Not through the origin (see the markdown cell above).
    """
    x = group["mean_z_ref"].to_numpy()
    y = group["mean_z_harm"].to_numpy()
    slope, intercept, r2, mae = summarize_ols(x, y)
    return pd.Series({"ols_slope": slope, "ols_intercept": intercept, "r2": r2, "mae": mae, "n_rows": len(x)})


preservation_by_pathology = (
    effect_size_by_pathology.groupby(["ratio", "method", "disease"])
    .apply(summarize_pathology_preservation, include_groups=False)
    .reset_index()
)
preservation_by_pathology.to_csv(os.path.join(OUT_BASE, "preservation_by_pathology.csv"), index=False)
print("preservation_by_pathology:", preservation_by_pathology.shape)


In [ ]:
# Top 15 (disease, metric, bundle) combinations by |pooled mean z_ref| at ratio 70.
RATIO_FOR_TOP_EFFECTS = 70
REF_METHOD_FOR_TOP_EFFECTS = "HC"  # z_ref does not depend on method; any method works.
TOP_EFFECTS_METHODS = ["HC", "MLP", "NO_FILTERING", "G_MAD", "PCA_MD"]

ref70 = effect_size_by_pathology[
    (effect_size_by_pathology["ratio"] == RATIO_FOR_TOP_EFFECTS)
    & (effect_size_by_pathology["method"] == REF_METHOD_FOR_TOP_EFFECTS)
].copy()
ref70["abs_mean_z_ref"] = ref70["mean_z_ref"].abs()
top15_keys = ref70.nlargest(15, "abs_mean_z_ref")[["disease", "metric", "bundle", "mean_z_ref", "n"]]

side_by_side = effect_size_by_pathology[
    (effect_size_by_pathology["ratio"] == RATIO_FOR_TOP_EFFECTS)
    & (effect_size_by_pathology["method"].isin(TOP_EFFECTS_METHODS))
].pivot_table(index=["disease", "metric", "bundle"], columns="method", values="mean_z_harm").reset_index()
side_by_side = side_by_side.rename(columns={method: f"mean_z_harm_{method}" for method in TOP_EFFECTS_METHODS})

top_effects_ratio70 = top15_keys.merge(side_by_side, on=["disease", "metric", "bundle"], how="left")
sort_order = top_effects_ratio70["mean_z_ref"].abs().sort_values(ascending=False).index
top_effects_ratio70 = top_effects_ratio70.loc[sort_order].reset_index(drop=True)
top_effects_ratio70.to_csv(os.path.join(OUT_BASE, "top_effects_ratio70.csv"), index=False)
top_effects_ratio70


## Headline tables (method x ratio)

`HC` is the ceiling achievable in this setup (see the supplementary section at the end of the notebook), not a target of 1: read the other methods relative to `HC` at the same ratio.


In [ ]:
ols_slope_pivot = preservation_by_method_ratio.pivot_table(index="method", columns="ratio", values="ols_slope")
method_order = [m for m in MAIN_METHODS if m in ols_slope_pivot.index] + [
    m for m in ols_slope_pivot.index if m not in MAIN_METHODS
]
ols_slope_pivot = ols_slope_pivot.loc[method_order].round(3)
print("OLS slope of mean_z_harm ~ mean_z_ref (patients), method x ratio:")
ols_slope_pivot


In [ ]:
ols_intercept_pivot = preservation_by_method_ratio.pivot_table(
    index="method", columns="ratio", values="ols_intercept"
).loc[method_order].round(3)
print("OLS intercept of mean_z_harm ~ mean_z_ref (patients), method x ratio:")
ols_intercept_pivot


In [ ]:
r2_pivot = preservation_by_method_ratio.pivot_table(
    index="method", columns="ratio", values="r2"
).loc[method_order].round(3)
print("R^2 of mean_z_harm ~ mean_z_ref (patients), method x ratio:")
r2_pivot


In [ ]:
mae_pivot = preservation_by_method_ratio.pivot_table(
    index="method", columns="ratio", values="mae"
).loc[method_order].round(3)
print("Mean absolute error |mean_z_harm - mean_z_ref| (patients), method x ratio:")
mae_pivot


## Plots


In [ ]:
def get_ordered_main_methods(available):
    """MAIN_METHODS, restricted to and ordered by what is actually available."""
    available_set = set(available)
    return [method for method in MAIN_METHODS if method in available_set]


def build_color_map(ordered_cols):
    """Return a consistent color map for the provided method columns."""
    col_colors = dict(BASE_COLOR_PALETTES)
    remaining = [col for col in ordered_cols if col not in col_colors]
    if remaining:
        palette = sns.color_palette(COLOR_PALETTE, len(remaining))
        col_colors.update(dict(zip(remaining, palette)))
    return col_colors


In [ ]:
# preservation_metrics_vs_ratio.png
ordered_methods = get_ordered_main_methods(preservation_by_method_ratio["method"].unique())
col_colors = build_color_map(ordered_methods)
ratios_sorted = sorted(preservation_by_method_ratio["ratio"].unique())

panels = [("ols_slope", "OLS slope", 1), ("r2", "R$^2$", None), ("mae", "MAE", 0)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (col, label, ref) in zip(axes, panels):
    for method in ordered_methods:
        sub = preservation_by_method_ratio[preservation_by_method_ratio["method"] == method].sort_values("ratio")
        ax.plot(sub["ratio"], sub[col], marker="o", color=col_colors[method], label=method, linewidth=2)
    if ref is not None:
        ax.axhline(y=ref, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(ratios_sorted)
    ax.set_xticklabels([f"{r}%" for r in ratios_sorted])
    ax.set_xlabel("Disease ratio")
    ax.set_ylabel(label)
    ax.set_title(label)

axes[-1].legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.suptitle("Preservation of patient deviation from CamCAN vs disease ratio (patients)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_FOLDER, "preservation_metrics_vs_ratio.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# scatter_zharm_vs_zref.png
scatter_ratios = [50, 80]
scatter_methods = ["NO_FILTERING", "MLP", "HC"]

fig, axes = plt.subplots(len(scatter_ratios), len(scatter_methods), figsize=(15, 10), sharex=True, sharey=True)
for row, ratio in enumerate(scatter_ratios):
    for col, method in enumerate(scatter_methods):
        ax = axes[row, col]
        sub = patients[(patients["ratio"] == ratio) & (patients["method"] == method)]
        x = sub["mean_z_ref"].to_numpy()
        y = sub["mean_z_harm"].to_numpy()

        ax.scatter(x, y, s=8, alpha=0.3, color=col_colors.get(method, "gray"))
        lims = [min(x.min(), y.min()) if len(x) else -1, max(x.max(), y.max()) if len(x) else 1]
        ax.plot(lims, lims, color="black", linestyle="--", linewidth=1, label="identity")

        if len(x):
            slope, intercept, r2, _ = summarize_ols(x, y)
            fit_x = np.array(lims)
            ax.plot(fit_x, slope * fit_x + intercept, color="blue", linewidth=1.5, label="OLS fit")
            annotation = f"slope = {slope:.2f}\nintercept = {intercept:.2f}\nR$^2$ = {r2:.2f}"
        else:
            annotation = "no data"
        ax.text(
            0.05, 0.95, annotation,
            transform=ax.transAxes, va="top", ha="left", fontsize=11,
            bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none"},
        )
        if row == 0:
            ax.set_title(method)
        if col == 0:
            ax.set_ylabel(f"ratio {ratio}%\nmean z (harmonized)")
        if row == len(scatter_ratios) - 1:
            ax.set_xlabel("mean z (reference)")

axes[0, 0].legend(loc="lower right", fontsize=9)
fig.suptitle("Patient mean z: harmonized vs reference (one point per site x metric x bundle)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_FOLDER, "scatter_zharm_vs_zref.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# preservation_by_pathology_ratio70.png
ratio70_pathology = preservation_by_pathology[
    (preservation_by_pathology["ratio"] == 70) & (preservation_by_pathology["disease"] != "HC")
]
diseases_sorted = sorted(ratio70_pathology["disease"].unique())
ordered_methods_path = get_ordered_main_methods(ratio70_pathology["method"].unique())
col_colors_path = build_color_map(ordered_methods_path)

x = np.arange(len(diseases_sorted))
group_width = 0.8
bar_width = group_width / max(len(ordered_methods_path), 1)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
panels = [("ols_slope", "OLS slope", 1), ("r2", "R$^2$", None)]
for ax, (col, label, ref) in zip(axes, panels):
    for idx, method in enumerate(ordered_methods_path):
        values = [
            ratio70_pathology[
                (ratio70_pathology["disease"] == disease) & (ratio70_pathology["method"] == method)
            ][col].mean()
            for disease in diseases_sorted
        ]
        pos = x - group_width / 2 + (idx + 0.5) * bar_width
        ax.bar(pos, values, width=bar_width * 0.9, color=col_colors_path[method], edgecolor="black", label=method)
    if ref is not None:
        ax.axhline(y=ref, color="black", linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(diseases_sorted)
    ax.set_ylabel(label)
    ax.set_title(label)

axes[-1].legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.suptitle("Preservation by pathology (disease ratio = 70%)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_FOLDER, "preservation_by_pathology_ratio70.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# variance_ratio_vs_ratio.png
variance_patients = variance_ratio_by_method_ratio[variance_ratio_by_method_ratio["group"] == "patients"]
ordered_methods_var = get_ordered_main_methods(variance_patients["method"].unique())
col_colors_var = build_color_map(ordered_methods_var)
ratios_sorted_var = sorted(variance_patients["ratio"].unique())

fig, ax = plt.subplots(figsize=(10, 6))
for method in ordered_methods_var:
    sub = variance_patients[variance_patients["method"] == method].sort_values("ratio")
    ax.plot(
        sub["ratio"], sub["median_sd_ratio"],
        marker="o", color=col_colors_var[method], label=method, linewidth=2,
    )
ax.axhline(y=1, color="black", linestyle="--", linewidth=1, label="perfectly preserved")
ax.set_xticks(ratios_sorted_var)
ax.set_xticklabels([f"{r}%" for r in ratios_sorted_var])
ax.set_xlabel("Disease ratio")
ax.set_ylabel("Median sd(z_harm) / sd(z_ref)")
ax.set_title("Variance preservation vs disease ratio (patients)")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_FOLDER, "variance_ratio_vs_ratio.png"), dpi=150, bbox_inches="tight")
plt.show()


## Supplementary: harmonization error vs age

Why doesn't even `HC` reach slope = 1, intercept = 0? The synthetic site bias injected into these sites is multiplicative on the raw metric values, so it does not only shift each subject's value - it also rescales that subject's covariate effects, including the linear age effect fitted by the CamCAN normative model. ComBat's location/scale model assumes the covariate effects (age, sex, handedness) are site-invariant and only removes a per-bundle location/scale shift; it has no term to undo a rescaling of the age effect itself.

Even the best-case correction available here (`HC`, fit on the healthy controls only) therefore leaves a residual, covariate-dependent error `err = z_harm - z_ref` that trends with age: it grows with age for FW and MD and shrinks with age for FA (see the table and plot below). Because older pathology groups (AD, MCI) and younger ones (TBI, SCHZ, ADHD) sit at different points on that residual age curve, some pathologies look "amplified" (slope > 1) and others "compressed" (slope < 1) even under the ideal `HC` fit. This is why `HC` should be read as the practical ceiling for these synthetic sites, and methods should be compared to what `HC` achieves at a given ratio rather than to an absolute slope of 1.

Computed on a subset only (site replicates 0-9 of ratios 3% and 80%, all 10 metrics, methods `HC`/`MLP`/`NO_FILTERING`) to keep the per-subject x bundle error table cheap to cache.


In [ ]:
AGE_ERROR_COLUMNS = [
    "ratio", "site_id", "metric", "method", "sid", "bundle", "disease", "age", "z_ref", "z_harm", "err",
]


def compute_age_error_site(ratio, site_id):
    """err = z_harm - z_ref per subject x bundle, for HC/MLP/NO_FILTERING, all metrics."""
    rows = []
    for metric in METRICS:
        gt_path = os.path.join(
            SITE_ROOT, "100_" + str(ratio), str(site_id), f"gt_train_100_{ratio}_{site_id}_{metric}.csv"
        )
        if not os.path.isfile(gt_path):
            continue
        gt = pd.read_csv(gt_path)
        gt = zscore_vs_camcan(gt, metric)
        gt_z = gt[["sid", "bundle", "disease", "age", "z"]].rename(columns={"z": "z_ref"})

        metric_dir = os.path.join(HARM_ROOT, "100_" + str(ratio), str(site_id), metric)
        for method_dir, method_name in METHOD_DIR_TO_NAME.items():
            if method_name not in AGE_DIAGNOSTIC_METHODS:
                continue
            harm_file = find_harmonized_file(os.path.join(metric_dir, method_dir))
            if harm_file is None:
                continue
            harm = pd.read_csv(harm_file)
            harm = zscore_vs_camcan(harm, metric)
            harm_z = harm[["sid", "bundle", "z"]].rename(columns={"z": "z_harm"})

            merged = gt_z.merge(harm_z, on=["sid", "bundle"], how="inner")
            if merged.empty:
                continue
            merged["err"] = merged["z_harm"] - merged["z_ref"]
            merged["ratio"] = ratio
            merged["site_id"] = site_id
            merged["metric"] = metric
            merged["method"] = method_name
            rows.append(merged[AGE_ERROR_COLUMNS])

    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=AGE_ERROR_COLUMNS)


if not FORCE_RECOMPUTE and os.path.isfile(AGE_ERROR_PATH):
    age_error = pd.read_csv(AGE_ERROR_PATH)
else:
    age_error_parts = Parallel(n_jobs=N_JOBS)(
        delayed(compute_age_error_site)(ratio, site_id)
        for ratio in AGE_DIAGNOSTIC_RATIOS
        for site_id in AGE_DIAGNOSTIC_SITE_IDS
    )
    age_error = pd.concat(age_error_parts, ignore_index=True)
    os.makedirs(OUT_BASE, exist_ok=True)
    age_error.to_csv(AGE_ERROR_PATH, index=False)

age_error["age_bin"] = pd.cut(age_error["age"], bins=AGE_BIN_EDGES, labels=AGE_BIN_LABELS, right=False)
n_sites_age = age_error[["ratio", "site_id"]].drop_duplicates().shape[0]
print(f"age_error_subset: {len(age_error):,} rows ({n_sites_age} sites x {age_error['metric'].nunique()} metrics x {age_error['method'].nunique()} methods).")


In [ ]:
# Mean error per age bin, pooling HC + patients (metric-resolved: used below for the plot/report).
age_bin_by_metric = (
    age_error.groupby(["ratio", "method", "metric", "age_bin"], observed=True)["err"]
    .agg(mean_err="mean", n="size")
    .reset_index()
)

# Mean error per age bin, pooling metrics (disease-resolved: supplementary detail).
age_bin_by_disease = (
    age_error.groupby(["ratio", "method", "age_bin", "disease"], observed=True)["err"]
    .agg(mean_err="mean", n="size")
    .reset_index()
)

print("Mean harmonization error (z_harm - z_ref) by age bin, ratio 80%, fa/md/fw:")
r80 = age_bin_by_metric[(age_bin_by_metric["ratio"] == 80) & (age_bin_by_metric["metric"].isin(["fa", "md", "fw"]))]
age_bin_pivot = r80.pivot_table(index=["method", "metric"], columns="age_bin", values="mean_err")
age_bin_pivot = age_bin_pivot.reindex(columns=AGE_BIN_LABELS).round(3)
age_bin_pivot


In [ ]:
print("Mean harmonization error (z_harm - z_ref) by age bin and disease, ratio 80% (all metrics pooled):")
disease_pivot = age_bin_by_disease[age_bin_by_disease["ratio"] == 80].pivot_table(
    index=["method", "disease"], columns="age_bin", values="mean_err"
)
disease_pivot = disease_pivot.reindex(columns=AGE_BIN_LABELS).round(3)
disease_pivot


In [ ]:
def summarize_age_slope(group):
    """Slope (per +10 years) and R^2 of err regressed on the continuous age."""
    x = group["age"].to_numpy()
    y = group["err"].to_numpy()
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    r2 = r2_score(y, reg.predict(x.reshape(-1, 1)))
    return pd.Series({"slope_per_decade": reg.coef_[0] * 10, "r2": r2, "n_rows": len(x)})


age_slope_by_metric = (
    age_error.groupby(["ratio", "method", "metric"])
    .apply(summarize_age_slope, include_groups=False)
    .reset_index()
)

print("Slope of harmonization error on age (per +10 years), ratio 80%, all metrics:")
age_slope_pivot = age_slope_by_metric[age_slope_by_metric["ratio"] == 80].pivot_table(
    index="method", columns="metric", values="slope_per_decade"
)
age_slope_pivot = age_slope_pivot[[m for m in METRICS if m in age_slope_pivot.columns]].round(4)
age_slope_pivot


In [ ]:
# error_vs_age_ratio80.png
age80 = age_bin_by_metric[
    (age_bin_by_metric["ratio"] == 80) & (age_bin_by_metric["metric"].isin(AGE_ERROR_PLOT_METRICS))
]
metric_colors = dict(zip(AGE_ERROR_PLOT_METRICS, sns.color_palette("tab10", len(AGE_ERROR_PLOT_METRICS))))

fig, axes = plt.subplots(1, len(AGE_DIAGNOSTIC_METHODS), figsize=(16, 5), sharey=True)
for ax, method in zip(axes, AGE_DIAGNOSTIC_METHODS):
    sub_method = age80[age80["method"] == method]
    for metric in AGE_ERROR_PLOT_METRICS:
        sub = sub_method[sub_method["metric"] == metric].set_index("age_bin").reindex(AGE_BIN_LABELS)
        ax.plot(AGE_BIN_LABELS, sub["mean_err"], marker="o", color=metric_colors[metric], label=metric, linewidth=2)
    ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
    ax.set_title(method)
    ax.set_xlabel("Age bin")

axes[0].set_ylabel("Mean error (z_harm - z_ref)")
axes[-1].legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.suptitle("Harmonization error vs age, disease ratio = 80% (HC + patients pooled)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_FOLDER, "error_vs_age_ratio80.png"), dpi=150, bbox_inches="tight")
plt.show()
